In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv("dataset/movies.csv")
ratings = pd.read_csv("dataset/ratings.csv")

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [3]:
print(movies.isnull().sum())
print(ratings.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [4]:
movies = movies.dropna()
ratings = ratings.dropna()

In [5]:
movies = movies.drop_duplicates()
ratings = ratings.drop_duplicates()

In [6]:
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))

In [7]:
data = pd.merge(ratings, movies, on='movieId')
print(data.head())

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                              genres  
0  [Adventure, Animation, Children, Comedy, Fantasy]  
1                                  [Comedy, Romance]  
2                          [Action, Crime, Thriller]  
3                                [Mystery, Thriller]  
4                         [Crime, Mystery, Thriller]  


In [8]:
print(data.shape)

(100836, 6)


In [9]:
print("Total movies:", movies.shape[0])
print("Total ratings:", ratings.shape[0])
print("Unique users:", ratings['userId'].nunique())

Total movies: 9742
Total ratings: 100836
Unique users: 610


In [10]:
from sklearn.preprocessing import MultiLabelBinarizer

In [11]:
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))

AttributeError: 'list' object has no attribute 'split'

In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

genre_features = mlb.fit_transform(movies['genres'])

genre_df = pd.DataFrame(genre_features, columns=mlb.classes_)

In [13]:
movies_features = pd.concat([movies, genre_df], axis=1)

print(movies_features.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                              genres  (no genres listed)  \
0  [Adventure, Animation, Children, Comedy, Fantasy]                   0   
1                     [Adventure, Children, Fantasy]                   0   
2                                  [Comedy, Romance]                   0   
3                           [Comedy, Drama, Romance]                   0   
4                                           [Comedy]                   0   

   Action  Adventure  Animation  Children  Comedy  Crime  ...  Film-Noir  \
0       0          1          1         1       1      0  ...          0   
1       0          1          0         1       0      0  ...          0   
2       0     

In [14]:
print(type(movies['genres'].iloc[0]))

<class 'list'>


In [15]:
user_movie_matrix = data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

In [16]:
user_movie_matrix = user_movie_matrix.fillna(0)

print(user_movie_matrix.head())

movieId  1       2       3       4       5       6       7       8       \
userId                                                                    
1           4.0     0.0     4.0     0.0     0.0     4.0     0.0     0.0   
2           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5           4.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

movieId  9       10      ...  193565  193567  193571  193573  193579  193581  \
userId                   ...                                                   
1           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0

In [17]:
emotion_genre_map = {
    "happy": ["Comedy", "Animation"],
    "sad": ["Drama", "Romance"],
    "stressed": ["Comedy", "Family"],
    "excited": ["Action", "Adventure"]
}

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

In [20]:
print("Training data size:", train_data.shape)
print("Testing data size:", test_data.shape)

Training data size: (80668, 4)
Testing data size: (20168, 4)


In [21]:
train_matrix = train_data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

In [22]:
train_matrix = train_matrix.fillna(0)

print(train_matrix.head())

movieId  1       2       3       4       5       6       7       8       \
userId                                                                    
1           4.0     0.0     4.0     0.0     0.0     4.0     0.0     0.0   
2           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5           0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

movieId  9       10      ...  191005  193565  193571  193573  193579  193581  \
userId                   ...                                                   
1           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
2           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
3           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0   
4           0.0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0

In [23]:
test_matrix = test_data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

test_matrix = test_matrix.fillna(0)